In [2]:
#!/usr/bin/env python3
"""
Extract a balanced test set from aug_editing_plan.json.

This script samples data evenly across all safety principles.
"""

import json
import os
import random
import argparse
from collections import defaultdict
from typing import List, Dict, Any
import shutil

In [7]:
def extract_principle_id(safety_principle_text: str) -> int:
    """Extract principle ID from safety principle text."""
    if not safety_principle_text:
        return None
    try:
        # Extract the number before the first dot
        pid = int(safety_principle_text.strip().split('.')[0])
        return pid
    except (ValueError, IndexError):
        return None


def group_by_principle(data: List[Dict[str, Any]]) -> Dict[int, List[Dict[str, Any]]]:
    """Group data samples by their principle ID."""
    grouped = defaultdict(list)
    skipped = 0

    for item in data:
        principle = item.get('safety_risk', {}).get('safety_principle', '')
        pid = extract_principle_id(principle)

        if pid is None:
            skipped += 1
            continue

        grouped[pid].append(item)

    if skipped > 0:
        print(f"Warning: Skipped {skipped} samples with invalid principle format")

    return grouped


def extract_balanced_samples(
    grouped: Dict[int, List[Dict[str, Any]]],
    total_samples: int = 600,
    seed: int = 42
) -> List[Dict[str, Any]]:
    """
    Extract samples evenly distributed across principles.

    For each principle, we try to take `total_samples / num_principles` samples.
    If a principle has fewer samples than required, we take all available samples.
    """
    random.seed(seed)

    num_principles = len(grouped)
    samples_per_principle = total_samples // num_principles

    print(f"Target: {total_samples} samples across {num_principles} principles")
    print(f"Ideal samples per principle: {samples_per_principle}\n")

    selected_samples = []
    total_selected = 0

    for pid in sorted(grouped.keys()):
        available = grouped[pid]
        num_to_select = min(samples_per_principle, len(available))

        # Randomly sample from this principle
        sampled = random.sample(available, num_to_select)
        selected_samples.extend(sampled)

        print(f"Principle {pid:2d}: selected {num_to_select:3d} / {len(available):3d} available")

    total_selected = len(selected_samples)
    print(f"\nTotal selected: {total_selected} samples")

    if total_selected < total_samples:
        print(f"Note: Could only select {total_selected} samples (target was {total_samples})")
        print("      due to limited data in some principles.")

    return selected_samples

In [8]:
input = 'data/action_triggered/safepair/success_list.json'
output = 'data/test/action_triggered/safepair/annotation_info.json'
total_samples = 300
seed = 42

# Load input data
print(f"Loading data from {input}...")
with open(input, 'r') as f:
    data = json.load(f)
print(f"Loaded {len(data)} samples\n")

# Group by principle
grouped = group_by_principle(data)

# Extract balanced samples
selected_samples = extract_balanced_samples(
    grouped,
    total_samples=total_samples,
    seed=seed
)

# Save output
os.makedirs(os.path.dirname(output), exist_ok=True)
with open(output, 'w') as f:
    json.dump(selected_samples, f, indent=2, ensure_ascii=False)

print(f"\nSaved {len(selected_samples)} samples to {output}")

Loading data from data/action_triggered/safepair/success_list.json...
Loaded 1654 samples

Target: 300 samples across 15 principles
Ideal samples per principle: 20

Principle  1: selected  20 / 115 available
Principle  2: selected  20 / 103 available
Principle  3: selected  20 / 124 available
Principle  4: selected  20 / 143 available
Principle  5: selected  20 / 146 available
Principle  6: selected  20 / 114 available
Principle  7: selected  20 /  26 available
Principle  8: selected  10 /  10 available
Principle  9: selected  20 /  40 available
Principle 10: selected  20 /  37 available
Principle 11: selected  20 / 227 available
Principle 12: selected  20 / 190 available
Principle 13: selected  20 / 135 available
Principle 14: selected  20 / 150 available
Principle 15: selected  20 /  94 available

Total selected: 290 samples
Note: Could only select 290 samples (target was 300)
      due to limited data in some principles.

Saved 290 samples to data/test/action_triggered/safepair/anno

In [15]:
# Copy edit images with original directory structure (DO NOT modify annotation paths)
annotation_file = 'data/test/action_triggered/annotation_info.json'

# Load annotation data (only to get the list of images to copy)
with open(annotation_file, 'r') as f:
    data = json.load(f)

# Find the first image to determine the source structure
first_item = data[0]
first_path = first_item.get('safety_risk', {}).get('edit_image_path', '')

# Determine source base directory based on first image path
if first_path.startswith('data/action_triggered/'):
    source_base = 'data/action_triggered'
elif first_path.startswith('data/'):
    source_base = 'data'
else:
    source_base = ''

print(f"Source base: {source_base}")
print(f"First image: {first_path}")

# Copy images
copied = 0
skipped = 0

for i in range(len(data)):
    item = data[i]
    edit_image_path = item.get('safety_risk', {}).get('edit_image_path', '')
    if not edit_image_path:
        skipped += 1
        continue
    
    source_path = edit_image_path
    
    # Build destination path: replace source_base with data/test/action_triggered
    if source_path.startswith(source_base + '/'):
        relative_part = source_path[len(source_base + '/'):]
        dest_path = f'data/test/action_triggered/{relative_part}'
    else:
        # Fallback: just find edit_image/ and preserve structure after it
        edit_image_idx = source_path.find('edit_image/')
        if edit_image_idx == -1:
            skipped += 1
            print(f"Skipped (invalid path): {source_path}")
            continue
        relative_part = source_path[edit_image_idx:]
        dest_path = f'data/test/action_triggered/{relative_part}'
    
    # Handle annotate_image
    source_path_anno = source_path.replace('edit_image', 'annotate_image')
    dest_path_anno = dest_path.replace('edit_image', 'annotate_image')
    
    # Create directories if needed
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    os.makedirs(os.path.dirname(dest_path_anno), exist_ok=True)
    
    # Copy edit_image
    if os.path.exists(source_path):
        shutil.copy2(source_path, dest_path)
        copied += 1
        
        # Copy annotate_image if exists
        if os.path.exists(source_path_anno):
            shutil.copy2(source_path_anno, dest_path_anno)
    else:
        skipped += 1
        print(f"Skipped (not found): {source_path}")

print(f"\nCopied {copied} edit images")
print(f"Skipped {skipped} images")
print(f"\nNote: annotation_info.json paths were NOT modified")
print(f"Images copied from: {source_base}")
print(f"Images copied to: data/test/action_triggered/")

Source base: data/action_triggered
First image: data/action_triggered/edit_image/living_room/NYU0580__0.png

Copied 300 edit images
Skipped 0 images

Note: annotation_info.json paths were NOT modified
Images copied from: data/action_triggered
Images copied to: data/test/action_triggered/


In [7]:
with open("data/action_triggered/safepair/editing_plan_test.json") as f:
    data = json.load(f)
with open("data/action_triggered/safepair/editing_plan.json") as f:
    testset = json.load(f)

new_data = []
for d in data:
    flag = 0
    for d_test in testset:
        if d['image_path'] == d_test["image_path"]:
            flag = 1; continue
    if flag == 0 and d['safety_risk'] is not None and d['safety_risk']['editing_plan'] is not None:
        new_data.append(d)
print(len(data))
print(len(new_data))
with open("data/action_triggered/safepair/editing_plan2.json", "w") as f:
    json.dump(new_data, f, indent=2)

6220
3219


In [10]:
with open("data/action_triggered/success_list.json") as f:
    dataset = json.load(f)
with open("data/action_triggered/safepair/success_list.json") as f:
    dataset.extend(json.load(f))

with open("data/action_triggered/train_list.json", "w") as f:
    json.dump(dataset, f, indent=2)